# Calculate RUL (corrected)

**Change from original:** added piecewise-linear RUL capping (Step 1 of the
ML performance checklist).

**Why:** raw RUL is unbounded (up to ~330+ cycles) and, for most of an
engine's life, sensor readings barely change while RUL still counts down
linearly from a large number. The model has no signal to distinguish
"280 cycles left" from "300 cycles left" — that stretch is unlearnable
variance that was inflating the loss and capping achievable R² regardless
of architecture. Capping RUL at 125 (the standard choice for CMAPSS FD001
in the literature) turns that unlearnable region into a constant, which is
the single biggest lever for improving validation R² on this dataset —
bigger than any architecture change.

The cap is applied identically to **both** the not-removed-sensor and
removed-sensor dataframes, so it propagates consistently through every
downstream notebook (splitting, scaling, sequence creation) without any
further changes needed there.


In [2]:
import pandas as pd

In [3]:
INPUT_PATH = "../CMAPSSData/processed/train_FD001_cleaned(removed_the_sensor).csv" 

In [4]:
df = pd.read_csv(INPUT_PATH)


In [5]:
df = df.sort_values(
    by=["unit_id", "cycle"]
).reset_index(drop=True)

In [6]:
max_cycle = df.groupby("unit_id")["cycle"].transform("max")

In [7]:
df["RUL"] = max_cycle - df["cycle"]

In [8]:
print(df[["unit_id", "cycle", "RUL"]].head())

   unit_id  cycle  RUL
0        1      1  191
1        1      2  190
2        1      3  189
3        1      4  188
4        1      5  187


In [9]:
print("\nNegative RUL:", (df["RUL"] < 0).sum())


Negative RUL: 0


## RUL capping (NEW)

Piecewise-linear cap at 125 cycles, applied to the raw RUL column before
anything else touches it — so every downstream file (split, scale,
sequence) sees the capped target and stays consistent between train and
validation.


In [ ]:
RUL_CAP = 130

print("Before capping:")
print("df1 RUL max:", df["RUL"].max(), " | > cap:", (df["RUL"] > RUL_CAP).sum(), "rows")

df["RUL"] = df["RUL"].clip(upper=RUL_CAP)

print("\nAfter capping:")
print("df1 RUL max:", df["RUL"].max())

Before capping:
df1 RUL max: 361  | > cap: 8031 rows

After capping:
df1 RUL max: 125


In [11]:
output_path = "../CMAPSSData/processed/train_FD001_cleaned_added_RUL(removed_the_sensor).csv"
df.to_csv(output_path, index=False)